In [1]:
import pandas as pd
import altair as alt
from pathlib import Path

In [2]:
result_dir = Path("../../clax-results/3-baidu-ultr-embeddings/")

In [3]:
df = pd.concat([pd.read_csv(f) for f in list(result_dir.glob("*/test_*.csv"))], ignore_index=True)
df.head()

,model,test_loss,test_ll,test_ppl,test_cond_ppl,train_time_s
0,DBN,0.245261,-0.243375,1.155766,1.148928,7034.878332
1,SDBN,0.295470,-0.293269,1.205800,1.222299,6725.317706
2,DCM,0.294107,-0.291969,1.212990,1.226355,3793.834832
3,CCM,0.245252,-0.243365,1.155770,1.148914,4206.636154
4,GCTR,0.297507,-0.295349,1.235433,1.235433,1264.753385


In [4]:
model2color = {
    "PBM": "#3182bd",
    "UBM": "#6baed6",
    "DBN": "#31a354",
    "SDBN": "#74c476",
    "CM": "#fd8d3c",
    "CCM": "#fdae6b",
    "DCM": "#fdd0a2",
    "DCTR": "#969696",
    "RCTR": "#969696",
    "GCTR": "#bdbdbd",
}
df["train_time_min"] = df["train_time_s"] / 60

In [5]:
model_means = df.groupby('model')['test_ppl'].mean().sort_values()
sorted_models = model_means.index.tolist()

base = alt.Chart(df, title="Perplexity", width=250, height=200)

bars = base.mark_bar().encode(
    x=alt.X("model", title="", sort=sorted_models).axis(labelAngle=45),
    y=alt.Y("mean(test_ppl)", title="").scale(zero=False, domain=(1.14, 1.26), clamp=True),
    color=alt.Color("model", title="Models", legend=None).scale(domain=list(model2color.keys()), range=list(model2color.values())),
)

errors = base.mark_errorbar(extent="ci", thickness=4).encode(
    x=alt.X("model", title="", sort=sorted_models),
    y=alt.Y("test_ppl", title="").scale(zero=False)
)

ppl_chart = (bars + errors)
ppl_chart

alt.LayerChart(...)

In [6]:
model_means = df.groupby('model')['test_cond_ppl'].mean().sort_values()
sorted_models = model_means.index.tolist()

base = alt.Chart(df, title="Conditional Perplexity", width=250, height=200)

bars = base.mark_bar().encode(
    x=alt.X("model", title="", sort=sorted_models).axis(labelAngle=45),
    y=alt.Y("mean(test_cond_ppl)", title="").scale(zero=False, domain=(1.14, 1.26), clamp=True),
    color=alt.Color("model", title="Models", legend=None).scale(domain=list(model2color.keys()), range=list(model2color.values())),
)

errors = base.mark_errorbar(extent="ci", thickness=4).encode(
    x=alt.X("model", title="", sort=sorted_models),
    y=alt.Y("test_cond_ppl", title="").scale(zero=False)
)

cond_ppl_chart = (bars + errors)
cond_ppl_chart

alt.LayerChart(...)

In [7]:
model_means = df.groupby('model')['train_time_min'].mean().sort_values()
sorted_models = model_means.index.tolist()

base = alt.Chart(df, title="Training Time (mins)", width=250, height=200)

bars = base.mark_bar().encode(
    x=alt.X("model", title="", sort=sorted_models).axis(labelAngle=45),
    y=alt.Y("mean(train_time_min)", title="").scale(domain=(0, 120)),
    color=alt.Color("model", title="Models", legend=None).scale(domain=list(model2color.keys()), range=list(model2color.values())),
)

errors = base.mark_errorbar(extent="ci", thickness=4).encode(
    x=alt.X("model", title="", sort=sorted_models),
    y=alt.Y("train_time_min", title="").scale(zero=False)
)

time_chart = (bars + errors)
time_chart

alt.LayerChart(...)

In [10]:
chart = ppl_chart | cond_ppl_chart | time_chart
chart

alt.HConcatChart(...)